In [6]:
import duckdb
import pandas as pd 
import json
import os
from pathlib import Path

## Section 1: Load a small sample of dataset
- We use DuckDB to peek at the data without loading everything 
- 200 records is enough to understand structure and spot issues

In [12]:
# Build path relative to this notebook's location 
# .parent goes up from notebooks/ to repo root
REPO_ROOT = Path.cwd().parent
META_PARQUET_PATH = REPO_ROOT / "data" / "processed" / "meta_All_Beauty.parquet"

print(f"Looking for file at {META_PARQUET_PATH}")
print(f"File exists: {META_PARQUET_PATH.exists()}")

Looking for file at /Users/jangaya/Desktop/MDS/Classes/block_6/575/DSCI_575_project_yasieft_purityj/data/processed/meta_All_Beauty.parquet
File exists: True


In [20]:
query = f"""
SELECT * 
FROM read_parquet('{META_PARQUET_PATH}')
LIMIT 200
"""

df_sample = duckdb.query(query).df()

print(f"Sample shape: {df_sample.shape}")
print(f"\nColumn types:\n{df_sample.dtypes}")
print(f"\nSample data:\n{df_sample.head()}")

Sample shape: (200, 16)

Column types:
main_category          str
title                  str
average_rating     float64
rating_number        int64
features            object
description         object
price                  str
images              object
videos              object
store                  str
categories          object
details                str
parent_asin            str
bought_together      Int32
subtitle             Int32
author               Int32
dtype: object

Sample data:
  main_category                                              title  \
0    All Beauty  Howard LC0008 Leather Conditioner, 8-Ounce (4-...   
1    All Beauty  Yes to Tomatoes Detoxifying Charcoal Cleanser ...   
2    All Beauty   Eye Patch Black Adult with Tie Band (6 Per Pack)   
3    All Beauty  Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...   
4    All Beauty  Precision Plunger Bars for Cartridge Grips – 9...   

   average_rating  rating_number  \
0             4.8             10   
1       

## Section 2 - Inspect individual records
- We want to see what the actual text/content of each record looks like

In [32]:
record = df_sample.iloc[5]
record

main_category                                             All Beauty
title              Lurrose 100Pcs Full Cover Fake Toenails Artifi...
average_rating                                                   3.7
rating_number                                                     35
features           [The false toenails are durable with perfect l...
description        [Description, The false toenails are durable w...
price                                                           6.99
images             {'hi_res': ['https://m.media-amazon.com/images...
videos                       {'title': [], 'url': [], 'user_id': []}
store                                                        Lurrose
categories                                                        []
details            {"Color": "As Shown", "Size": "Large", "Materi...
parent_asin                                               B07G9GWFSM
bought_together                                                 <NA>
subtitle                          

In [36]:
print("=== TITLE ===")
print(record['title'])

print("\n=== FEATURES ===")
# features is a list - print each bullet point separately
for f in record['features']:
    print(f"  • {f}")

print("\n=== DESCRIPTION ===")
for d in record['description']:
    print(f"  {d}")

print("\n=== DETAILS ===")
print(record['details'])

print("\n=== PRICE ===")
print(record['price'])


=== TITLE ===
Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY

=== FEATURES ===
  • The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like.
  • ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails.
  • Fit well to your natural toenails. Non toxic, no smell, no harm to your health.
  • Wonderful as gift for girlfriend, family and friends.
  • The easiest and most efficient way to do your toenail tips for manicures or nail art designs. It's fashion, creative, a useful accessory brighten up your look, also as a gift.

=== DESCRIPTION ===
  Description
  The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. Plus, ABS is kind of green enviromen

## Section 3 - Check for missing data
- Missing fields = gaps in our searchable document
- Examine how often each field is missing across ALL records
- Use full dataset here via DuckDB - no need to laod into RAM

In [38]:
missing = duckdb.query(f"""
    SELECT
        -- COUNT checks how many records have non-null values
        -- We subtract from total to get missing count
        COUNT(*) as total,
        COUNT(*) - COUNT(title) as missing_title,
        COUNT(*) - COUNT(price) as missing_price,
        COUNT(*) - COUNT(store) as missing_store,
        COUNT(*) - COUNT(average_rating) as missing_rating,

        -- For array fields, check for empty arrays too
        SUM(CASE WHEN features = [] THEN 1 ELSE 0 END) as empty_features,
        SUM(CASE WHEN description = [] THEN 1 ELSE 0 END) as empty_description
    FROM read_parquet('{META_PARQUET_PATH}')
""").df()

print("Missing data across full 112K dataset:")
print(missing.T)  # .T transposes for easier reading

Missing data across full 112K dataset:
                          0
total              112590.0
missing_title           0.0
missing_price           0.0
missing_store       11331.0
missing_rating          0.0
empty_features      95213.0
empty_description   93428.0


## Section 4 - Investigate price field
- Check what values are in price field and why its stored as VARCHAR

In [40]:
price_analysis = duckdb.query(f"""
SELECT 
price,
COUNT(*) as count 
FROM read_parquet('{META_PARQUET_PATH}')
GROUP BY price
ORDER BY count DESC
LIMIT 20
""").df()

print("Most common price values:")
print(price_analysis)

Most common price values:
    price  count
0    None  94886
1    9.99    631
2   19.99    363
3   14.99    314
4    8.99    283
5    7.99    280
6    6.99    275
7   11.99    252
8   12.99    252
9    5.99    243
10  29.99    198
11  15.99    185
12  13.99    183
13  24.99    174
14  10.99    169
15  16.99    165
16  18.99    134
17   4.99    130
18  17.99    120
19   3.99    119


## Section 5 - Document contruction decision 
- Decide what 'Searchable documents' looks like before building any index

In [52]:
def build_document(row):
    """
    Combines relevant text fields into a single searchable string. 
    This is what BM25 and the embedding model will actually index.

    We combine: title + features + description + details
    - title: concise product name
    - features: bullet points with key product attributes  
    - description: longer form product context
    - details: brand, material, specific attributes
    """

    parts = []

    # Title - most important signal, always try to include it
    if pd.notna(row['title']) and row['title']:
        parts.append(str(row['title']))

    # Features - check length instead of type to handle numpy arrays too
    try:
        if row['features'] is not None and len(row['features']) > 0:
            parts.append(" ".join(str(f) for f in row['features']))
    except (TypeError, ValueError):
        pass

    # Description - same type-agnostic approach as features
    try:
        if row['description'] is not None and len(row['description']) > 0:
            parts.append(" ".join(str(d) for d in row['description']))
    except (TypeError, ValueError):
        pass

    # Store - captures brand names not always in details
    if pd.notna(row['store']) and row['store']:
        parts.append(str(row['store']))

    # Details - brand, material, skin type etc. stored as JSON string
    if pd.notna(row['details']) and row['details']:
        parts.append(str(row['details']))

    return " ".join(parts)

# apply to our sample to see what documents look like
df_sample['document'] = df_sample.apply(build_document, axis=1)
df_sample.head()


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,document
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],None,"{'hi_res': [None, 'https://m.media-amazon.com/...","{'title': [], 'url': [], 'user_id': []}",Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",B01CUPMQZE,<NA>,<NA>,<NA>,"Howard LC0008 Leather Conditioner, 8-Ounce (4-..."
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",B076WQZGPM,<NA>,<NA>,<NA>,Yes to Tomatoes Detoxifying Charcoal Cleanser ...
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],None,"{'hi_res': [None, None], 'large': ['https://m....","{'title': [], 'url': [], 'user_id': []}",Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,<NA>,<NA>,<NA>,Eye Patch Black Adult with Tie Band (6 Per Pac...
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",B088FKY3VD,<NA>,<NA>,<NA>,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4..."
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Precision,[],"{""UPC"": ""644287689178""}",B07NGFDN6G,<NA>,<NA>,<NA>,Precision Plunger Bars for Cartridge Grips – 9...


In [53]:
# Inspect for a few documents
for i in range(3):
    print(f"\n=== Document {i+1} ===")
    print(f"Title: {df_sample.iloc[i]['title']}")
    print(f"Document length: {len(df_sample.iloc[i]['document'])} chars")
    print(f"Document preview: {df_sample.iloc[i]['document'][:300]}...")
    print("-" * 50)
    


=== Document 1 ===
Title: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)
Document length: 150 chars
Document preview: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack) Howard Products {"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}...
--------------------------------------------------

=== Document 2 ===
Title: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.
Document length: 422 chars
Document preview: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz. Yes To {"Item Form": "Powder", "Skin Type": "Acne Prone", "Brand": "Yes To", "Age Range (Description)": "Adult", "Unit Count": "10 Fl Oz", "Is Discontinued ...
--------------------------------------------------

=== Document 3 ===
Title: Eye Patch Black Adult with Tie Band (6 Per Pack)
Document length: 114 cha

In [ ]:
# check for the bug in build_document
# row = df_sample.iloc[4]
# print(f"features type: {type(row['features'])}")
# print(f"features value: {row['features']}")
# print(f"is list: {isinstance(row['features'], list)}")

In [56]:
# Print the full document for record index 4
# This is our "best case" - a product with all fields populated
print("=== Full Document 5 (has features + description) ===")
print(df_sample.iloc[4]['document'])
print(f"\nLength: {len(df_sample.iloc[4]['document'])} chars")

=== Full Document 5 (has features + description) ===
Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers Material: 304 Stainless Steel; Brass tip Lengths Available: 88mm, 93mm, 98mm Accepts cartridge needles with vice style tattoo machines Works perfectly with Precision Disposable Soft Cartridge Grips Price per one bag of 10 plungers The Precision Plunger Bars are designed to work seamlessly with the Precision Disposable 1. 25" Contoured Soft Cartridge Grips and the Precision Disposable 1" Textured Soft Cartridge Grips to drive cartridge needles with vice style or standard tattoo machine setups. These plunger bars are manufactured from 304 Stainless Steel and feature a brass tip. The plungers are sold in a bag of ten in your choice of 88mm, 93mm, or 98mm length. Precision {"UPC": "644287689178"}

Length: 772 chars


In [55]:
# Check if store information is already captured in details
# Look at 5 records side by side
for i in range(5):
    row = df_sample.iloc[i]
    print(f"\nProduct: {row['title'][:50]}")
    print(f"  store:   {row['store']}")
    # details often has Brand field - check if it matches store
    print(f"  details: {str(row['details'])[:100]}")


Product: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack
  store:   Howard Products
  details: {"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}

Product: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pac
  store:   Yes To
  details: {"Item Form": "Powder", "Skin Type": "Acne Prone", "Brand": "Yes To", "Age Range (Description)": "Ad

Product: Eye Patch Black Adult with Tie Band (6 Per Pack)
  store:   Levine Health Products
  details: {"Manufacturer": "Levine Health Products"}

Product: Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4D Im
  store:   Cherioll
  details: {"Brand": "Cherioll", "Item Form": "Powder", "Finish Type": "Natural", "Product Benefits": "Long Las

Product: Precision Plunger Bars for Cartridge Grips – 93mm 
  store:   Precision
  details: {"UPC": "644287689178"}


In [57]:
# Get accurate statistics on the full 112K dataset
# Run directly on parquet via DuckDB - no RAM concerns
full_stats = duckdb.query(f"""
    SELECT
        COUNT(*) as total,

        -- Text field emptiness
        SUM(CASE WHEN title IS NULL OR title = '' 
            THEN 1 ELSE 0 END) as missing_title,
        SUM(CASE WHEN features = [] 
            THEN 1 ELSE 0 END) as empty_features,
        SUM(CASE WHEN description = [] 
            THEN 1 ELSE 0 END) as empty_description,
        SUM(CASE WHEN store IS NULL OR store = '' 
            THEN 1 ELSE 0 END) as missing_store,

        -- Price: check for both NULL and the string "None"
        SUM(CASE WHEN price IS NULL OR price = 'None' 
            THEN 1 ELSE 0 END) as missing_price,

        -- Document richness: how many have ONLY title+details
        SUM(CASE WHEN features = [] AND description = [] 
            THEN 1 ELSE 0 END) as title_only_products

    FROM read_parquet('{META_PARQUET_PATH}')
""").df()

# Show as percentages for easy interpretation
total = full_stats['total'][0]
for col in full_stats.columns[1:]:
    count = full_stats[col][0]
    pct = count / total * 100
    print(f"{col:25s}: {count:6.0f} / {total:.0f}  ({pct:.1f}%)")

missing_title            :     12 / 112590  (0.0%)
empty_features           :  95213 / 112590  (84.6%)
empty_description        :  93428 / 112590  (83.0%)
missing_store            :  11331 / 112590  (10.1%)
missing_price            :  94886 / 112590  (84.3%)
title_only_products      :  88746 / 112590  (78.8%)


## EDA Key Findings

### Dataset: All_Beauty
- Total products (metadata): 112,590
- Total reviews: 701,528

### 1. Sparse text fields
| Field | Missing/Empty | Percentage |
|---|---|---|
| title | 12 | 0.0% |
| features | 95,213 | 84.6% |
| description | 93,428 | 83.0% |
| store | 11,331 | 10.1% |
| price | 94,886 | 84.3% |

78.8% of products (88,746) have ONLY title + details available 
for retrieval — no features or description at all.

### 2. Price unusable for filtering
84.3% of prices are missing. Price is stored as VARCHAR 
containing the string "None" rather than SQL NULL — a data 
quality issue that requires explicit string checking.

### 3. Document construction
We combine title + features + description + store + details 
into a single searchable string per product. Document length 
varies widely:
- Thin document (title + details only): ~91–134 chars (78.8% of products)
- Rich document (all fields populated): ~400–800 chars (21.2% of products)

### 4. Type inconsistency in array fields
features and description load as numpy arrays from Parquet, 
not Python lists. isinstance(x, list) checks fail silently. 
Fixed using type-agnostic len() checks.

### 5. Store field adds missing brand signal
store captures brand names not always present in details.
10.1% of products have no store value.

### 6. Implications for retrieval
- BM25 will handle exact keyword queries reasonably 
  (title always present)
- Both methods will struggle with descriptive queries on 
  the 78.8% of thin documents
- Price filtering not feasible due to 84.3% missing values
- Dataset sparsity is the primary limitation of this system

---

## Decisions

| Decision | Rationale |
|---|---|
| Document = title + features + description + store + details | Most descriptive text fields available |
| Exclude price from document | 84.3% missing — adds noise not signal |
| Include store | Captures brand names missing from details (10% of products) |
| Type-agnostic checks with len() | Parquet loads arrays as numpy — isinstance(x, list) fails silently |
| Use full 112,590 products | Small enough to index entirely on any laptop |

### Verify that utils.py works correctly

In [1]:
import sys
sys.path.append('..')  # lets notebook import from src/

from src.utils import load_corpus, tokenize, build_document

# Test tokenize
print("=== Tokenizer test ===")
test_text = "Gentle Moisturizer for Sensitive Skin with Vitamin C"
tokens = tokenize(test_text)
print(f"Input:  {test_text}")
print(f"Tokens: {tokens}")

# Test load_corpus on small subset first
print("\n=== Corpus loading test (100 records) ===")
documents, metadata = load_corpus(limit=100)
print(f"Documents: {len(documents)}")
print(f"\nSample document:\n{documents[4][:300]}")
print(f"\nSample metadata:\n{metadata[4]}")

=== Tokenizer test ===
Input:  Gentle Moisturizer for Sensitive Skin with Vitamin C
Tokens: ['gentle', 'moisturizer', 'sensitive', 'skin', 'vitamin']

=== Corpus loading test (100 records) ===
Loaded 100 documents from corpus
Documents: 100

Sample document:
Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers Material: 304 Stainless Steel; Brass tip Lengths Available: 88mm, 93mm, 98mm Accepts cartridge needles with vice style tattoo machines Works perfectly with Precision Disposable Soft Cartridge Grips Price per one bag of 10 plunger

Sample metadata:
{'parent_asin': 'B07NGFDN6G', 'title': 'Precision Plunger Bars for Cartridge Grips – 93mm – Bag of 10 Plungers', 'price': 'None', 'average_rating': 4.3, 'rating_number': 7, 'store': 'Precision'}
